# Notebook 01 - Veri Kesfi

Bu notebook kapsaminda Amasya ilindeki Hamamozu, Gumushacikoy ve Goynucek ilcelerine ait elektrik tuketim ve tahsilat verileri incelenmektedir.

## Icindekiler

1. Veri Setinin Yuklenmesi
2. Veri Seti Genel Ozeti
2.1. Temel Pandas Kontrolleri
3. Eksik Deger Kontrolu
4. Tahakkuk Verilerinin Birlestirilmesi
5. Ilce Bazinda Benzersiz Musteri Sayilari
6. kWh Tuketim Verisinin Kalite Kontrolu
7. Aykiri Deger Analizi
8. Hesap Sinifina Gore Tuketim Istatistikleri
9. Analiz Kapsami Kontrolu
10. Analize Hazirlik
11. Genel Degerlendirme

### Amaclar
- Veri setlerinin yapisini ve temel ozelliklerini ozetlemek
- Eksik deger, veri tipi ve yinelenen kayit kontrollerini yapmak
- Ilce bazinda benzersiz musteri sayilarini karsilastirmak
- Tahakkuk verilerini tek bir veri setinde birlestirmek
- kWh tuketim verilerindeki negatif, sifir ve aykiri degerleri incelemek
- Hesap siniflarina gore tuketim istatistiklerini analiz etmek


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Veri Setinin Yuklenmesi

Excel dosyasinda bulunan tahsilat ve tahakkuk sayfalari ayri DataFrame'lere yuklenmistir.


In [2]:
veri_dosyasi_adaylari = [
    Path("../data/elektrik_veri.xlsx"),
    Path("data/elektrik_veri.xlsx"),
    Path("case_study02/data/elektrik_veri.xlsx"),
]

file_path = next(path for path in veri_dosyasi_adaylari if path.exists())

xls = pd.ExcelFile(file_path)

print("Excel sayfalari:")
print(xls.sheet_names)


Excel sayfalari:
['Tahsilat', 'Tahsilat 1', 'Tahakkuk', 'Tahakkuk 1', 'Tahakkuk 2']


In [3]:
df_tahsilat = pd.read_excel(xls, sheet_name="Tahsilat")
df_tahsilat_1 = pd.read_excel(xls, sheet_name="Tahsilat 1")

df_hamamozu = pd.read_excel(xls, sheet_name="Tahakkuk")
df_gumushacikoy = pd.read_excel(xls, sheet_name="Tahakkuk 1")
df_goynucek = pd.read_excel(xls, sheet_name="Tahakkuk 2")

dataframes = {
    "Tahsilat": df_tahsilat,
    "Tahsilat 1": df_tahsilat_1,
    "Hamamozu": df_hamamozu,
    "Gumushacikoy": df_gumushacikoy,
    "Goynucek": df_goynucek,
}


## 2. Veri Seti Genel Ozeti

Tum veri setleri icin satir/sutun sayisi, eksik deger durumu ve degisken tipleri tek tabloda ozetlenmistir. Boylece notebook ciktisi gereksiz uzun `info()` ve `describe()` bloklariyla kalabaliklasmaz.


In [4]:
veri_ozeti = []

for name, df in dataframes.items():
    veri_ozeti.append({
        "Veri Seti": name,
        "Satir Sayisi": len(df),
        "Sutun Sayisi": df.shape[1],
        "Toplam Eksik Deger": df.isna().sum().sum(),
        "Sayisal Sutun": df.select_dtypes(include="number").shape[1],
        "Tarih Sutunu": df.select_dtypes(include="datetime").shape[1],
        "Metin/Kategorik Sutun": df.select_dtypes(include=["object", "string"]).shape[1],
    })

df_veri_ozeti = pd.DataFrame(veri_ozeti)
display(df_veri_ozeti)


,Veri Seti,Satir Sayisi,Sutun Sayisi,Toplam Eksik Deger,Sayisal Sutun,Tarih Sutunu,Metin/Kategorik Sutun
0,Tahsilat,636993,9,1910974,5,1,3
1,Tahsilat 1,917632,22,13478526,18,0,4
2,Hamamozu,124818,10,0,2,0,8
3,Gumushacikoy,765657,10,0,2,0,8
4,Goynucek,295223,10,0,2,0,8


In [5]:
kolon_ozeti = pd.DataFrame({
    "Veri Seti": list(dataframes.keys()),
    "Kolonlar": [", ".join(df.columns.astype(str)) for df in dataframes.values()]
})

display(kolon_ozeti)


,Veri Seti,Kolonlar
0,Tahsilat,"Şube, Kasa, İlçe, Söz.hsp.(bağımsız), Tahsilat..."
1,Tahsilat 1,"Mali yıl/dönem, İl, İlçe, Söz.hsp.(bağımsız), ..."
2,Hamamozu,"il, ilce, sozlesme_hesap_no, mali_yil_donem, f..."
3,Gumushacikoy,"il, ilce, sozlesme_hesap_no, mali_yil_donem, f..."
4,Goynucek,"il, ilce, sozlesme_hesap_no, mali_yil_donem, f..."


## 2.1. Temel Pandas Kontrolleri

Yonergede belirtilen `.head()`, `.info()` ve `.describe()` kontrolleri her DataFrame icin uygulanmistir. Cikti okunabilirligini korumak icin kontroller veri seti adlariyla bolumlenmistir.


In [6]:
for name, df in dataframes.items():
    print(f"\n{name} - head()")
    display(df.head())



Tahsilat - head()


,Şube,Kasa,İlçe,Söz.hsp.(bağımsız),Tahsilat Tarihi,Nakit Tahsilat,Mahsuben Tahsilat,Kredi Kartı Tahsilatı,Banka Tahsilatı
0,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2023-11-06,NaN,"8,648.95",NaN,NaN
1,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-06-26,NaN,762.40,NaN,NaN
2,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-07-10,NaN,311.60,NaN,NaN
3,PTT,PTT/PV,TAŞOVA,4254955886,2023-01-19,NaN,NaN,NaN,130.50
4,PTT,PTT/PV,TAŞOVA,4254955886,2023-02-17,NaN,NaN,NaN,117.00



Tahsilat 1 - head()


,Mali yıl/dönem,İl,İlçe,Söz.hsp.(bağımsız),Hesap Sınıfı,Tahakkuk Tutar,Son Ödeme Tarihinden Önceki Tahsilat,Son Ödeme Tarihindeki Tahsilat,Son Ödeme (1),Son Ödeme (2),Son Ödeme (3),Son Ödeme (4),Son Ödeme (5),Son Ödeme (6-10),Son Ödeme (10-20),Son Ödeme (20-30),Son Ödeme (30-60),Son Ödeme (60-90),Son Ödeme (90-120),Son Ödeme (120-150),Son Ödeme (150-180),Son Ödeme (180+)
0,OCK 2023,AMASYA,GÖYNÜCEK,9374624783,Mesken,5.03,0.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.00,NaN,NaN,NaN
1,OCK 2023,AMASYA,GÖYNÜCEK,236184905,Mesken,26.46,0.06,26.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OCK 2023,AMASYA,GÖYNÜCEK,9657731015,Mesken,121.53,NaN,NaN,NaN,121.53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,OCK 2023,AMASYA,GÖYNÜCEK,9554442880,Mesken,117.49,NaN,117.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,OCK 2023,AMASYA,GÖYNÜCEK,6031642522,Mesken,170.30,170.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Hamamozu - head()


,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
0,AMASYA,HAMAMÖZÜ,917576806,2023-01-01,2023-01-12,2023-03-06,2023-01-23,M001,Mesken,1.79
1,AMASYA,HAMAMÖZÜ,917576806,2023-01-01,2023-02-09,2023-05-11,2023-02-20,M001,Mesken,2.60
2,AMASYA,HAMAMÖZÜ,917576806,2023-02-01,2023-02-09,2023-05-11,2023-02-20,M001,Mesken,1.23
3,AMASYA,HAMAMÖZÜ,917576806,2023-02-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,2.56
4,AMASYA,HAMAMÖZÜ,917576806,2023-03-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,1.35



Gumushacikoy - head()


,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
0,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-01-01,2023-01-11,2023-03-06,2023-01-23,M001,Mesken,21.85
1,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-01-01,2023-02-10,2023-05-11,2023-02-20,M001,Mesken,44.50
2,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-02-01,2023-02-10,2023-05-11,2023-02-20,M001,Mesken,22.25
3,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-02-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,45.71
4,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-03-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,25.40



Goynucek - head()


,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
0,AMASYA,GÖYNÜCEK,9374624783,2023-01-01,2023-01-14,2023-03-06,2023-01-24,M001,Mesken,0.10
1,AMASYA,GÖYNÜCEK,9374624783,2023-01-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.12
2,AMASYA,GÖYNÜCEK,9374624783,2023-02-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.20
3,AMASYA,GÖYNÜCEK,9374624783,2023-03-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.00
4,AMASYA,GÖYNÜCEK,9374624783,2023-04-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.08


In [7]:
for name, df in dataframes.items():
    print(f"\n{name} - info()")
    df.info()



Tahsilat - info()
<class 'pandas.DataFrame'>
RangeIndex: 636993 entries, 0 to 636992
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Şube                   636993 non-null  str           
 1   Kasa                   636993 non-null  str           
 2   İlçe                   636993 non-null  str           
 3   Söz.hsp.(bağımsız)     636993 non-null  int64         
 4   Tahsilat Tarihi        636993 non-null  datetime64[us]
 5   Nakit Tahsilat         523 non-null     float64       
 6   Mahsuben Tahsilat      7542 non-null    float64       
 7   Kredi Kartı Tahsilatı  0 non-null       float64       
 8   Banka Tahsilatı        628933 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(3)
memory usage: 43.7 MB

Tahsilat 1 - info()
<class 'pandas.DataFrame'>
RangeIndex: 917632 entries, 0 to 917631
Data columns (total 22 columns):
 #   Column            

In [8]:
for name, df in dataframes.items():
    print(f"\n{name} - describe() sayisal")
    display(df.describe())

    kategorik_df = df.select_dtypes(include=["object", "string"])
    if not kategorik_df.empty:
        print(f"{name} - describe() kategorik")
        display(kategorik_df.describe())



Tahsilat - describe() sayisal


,Söz.hsp.(bağımsız),Tahsilat Tarihi,Nakit Tahsilat,Mahsuben Tahsilat,Kredi Kartı Tahsilatı,Banka Tahsilatı
count,"636,993.00",636993,523.00,"7,542.00",0.00,"628,933.00"
mean,"5,019,884,374.88",2024-03-05 09:26:07.644856,694.97,"6,180.18",NaN,372.63
min,"175,832.00",2023-01-01 00:00:00,7.45,"-34,508.95",NaN,0.01
25%,"2,532,887,116.00",2023-07-28 00:00:00,425.33,44.48,NaN,120.00
50%,"5,008,502,097.00",2024-02-26 00:00:00,524.67,290.41,NaN,208.00
75%,"7,525,722,213.00",2024-09-30 00:00:00,688.83,"2,729.11",NaN,322.00
max,"9,999,599,851.00",2025-05-31 00:00:00,"11,373.74","399,526.78",NaN,"606,473.80"
std,"2,884,434,992.84",NaN,758.32,"23,828.02",NaN,"3,265.43"


Tahsilat - describe() kategorik


,Şube,Kasa,İlçe
count,636993,636993,636993
unique,26,76,4
top,AKTİFBANK,AKT/PN,TAŞOVA
freq,193135,168183,289077



Tahsilat 1 - describe() sayisal


...

,il,ilce,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı
count,765657,765657,765657,765657,765657,765657,765657,765657
unique,1,1,31,758,30,645,36,36
top,AMASYA,GÜMÜŞHACIKÖY,2024-07-01,2025-06-21,2024-10-08,2024-07-01,M001,Mesken
freq,765657,765657,26246,15746,56751,16882,658189,658189



Goynucek - describe() sayisal


,sozlesme_hesap_no,kwh
count,"295,223.00","295,223.00"
mean,"4,950,566,464.43",89.67
std,"2,887,724,120.17",742.28
min,"1,118,550.00","-4,208.64"
25%,"2,432,854,136.00",17.86
50%,"4,928,206,069.00",45.09
75%,"7,475,616,412.00",77.14
max,"9,997,184,778.00","105,687.69"


Goynucek - describe() kategorik


,il,ilce,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı
count,295223,295223,295223,295223,295223,295223,295223,295223
unique,1,1,31,663,30,566,29,29
top,AMASYA,GÖYNÜCEK,2024-07-01,2024-07-16,2024-10-08,2023-12-25,M001,Mesken
freq,295223,295223,10113,9556,21865,10392,257738,257738


## 3. Eksik Deger Kontrolu

Eksik degerler veri seti bazinda incelenmistir. Tahsilat verilerindeki bazi eksik alanlar odeme turu veya odeme zaman araliginda tutar bulunmadigini temsil etmektedir.


In [9]:
def eksik_deger_ozeti(df, veri_adi):
    eksik = df.isna().sum()
    oran = (eksik / len(df) * 100).round(2)

    ozet = pd.DataFrame({
        "Veri Seti": veri_adi,
        "Sutun": eksik.index,
        "Eksik Deger Sayisi": eksik.values,
        "Eksik Deger Orani (%)": oran.values,
    })

    return ozet[ozet["Eksik Deger Sayisi"] > 0]

for name, df in dataframes.items():
    ozet = eksik_deger_ozeti(df, name)
    print(f"\n{name}")
    print("-" * 50)
    if ozet.empty:
        print("Eksik deger bulunmamaktadir.")
    else:
        display(ozet)



Tahsilat
--------------------------------------------------


,Veri Seti,Sutun,Eksik Deger Sayisi,Eksik Deger Orani (%)
5,Tahsilat,Nakit Tahsilat,636470,99.92
6,Tahsilat,Mahsuben Tahsilat,629451,98.82
7,Tahsilat,Kredi Kartı Tahsilatı,636993,100.00
8,Tahsilat,Banka Tahsilatı,8060,1.27



Tahsilat 1
--------------------------------------------------


,Veri Seti,Sutun,Eksik Deger Sayisi,Eksik Deger Orani (%)
6,Tahsilat 1,Son Ödeme Tarihinden Önceki Tahsilat,293724,32.01
7,Tahsilat 1,Son Ödeme Tarihindeki Tahsilat,589439,64.23
8,Tahsilat 1,Son Ödeme (1),896730,97.72
9,Tahsilat 1,Son Ödeme (2),895968,97.64
10,Tahsilat 1,Son Ödeme (3),898739,97.94
11,Tahsilat 1,Son Ödeme (4),900637,98.15
12,Tahsilat 1,Son Ödeme (5),910309,99.20
13,Tahsilat 1,Son Ödeme (6-10),871924,95.02
14,Tahsilat 1,Son Ödeme (10-20),869351,94.74
15,Tahsilat 1,Son Ödeme (20-30),888627,96.84



Hamamozu
--------------------------------------------------
Eksik deger bulunmamaktadir.

Gumushacikoy
--------------------------------------------------
Eksik deger bulunmamaktadir.

Goynucek
--------------------------------------------------
Eksik deger bulunmamaktadir.


## 4. Tahakkuk Verilerinin Birlestirilmesi

Uc ilceye ait tahakkuk verileri tek bir DataFrame altinda birlestirilmis ve birlestirme sonrasi kayit sayisi dogrulanmistir.


In [10]:
df_tahakkuk_all = pd.concat(
    [df_hamamozu, df_gumushacikoy, df_goynucek],
    ignore_index=True
)

toplam_beklenen = len(df_hamamozu) + len(df_gumushacikoy) + len(df_goynucek)

print(f"Hamamozu kayit sayisi     : {len(df_hamamozu):,}")
print(f"Gumushacikoy kayit sayisi : {len(df_gumushacikoy):,}")
print(f"Goynucek kayit sayisi     : {len(df_goynucek):,}")
print("-" * 40)
print(f"Beklenen toplam kayit      : {toplam_beklenen:,}")
print(f"Birlestirilmis kayit sayisi: {len(df_tahakkuk_all):,}")
print("Kayit sayisi dogrulamasi   :", len(df_tahakkuk_all) == toplam_beklenen)


Hamamozu kayit sayisi     : 124,818
Gumushacikoy kayit sayisi : 765,657
Goynucek kayit sayisi     : 295,223
----------------------------------------
Beklenen toplam kayit      : 1,185,698
Birlestirilmis kayit sayisi: 1,185,698
Kayit sayisi dogrulamasi   : True


In [11]:
tarih_sutunlari = [
    "mali_yil_donem",
    "fatura_tarihi",
    "kayit_tarihi",
    "vade_tarihi",
]

for column in tarih_sutunlari:
    df_tahakkuk_all[column] = pd.to_datetime(df_tahakkuk_all[column], errors="coerce")

hesap_sinifi_col = df_tahakkuk_all.columns[8]

df_tahakkuk_all["ilce"] = df_tahakkuk_all["ilce"].str.strip()
df_tahakkuk_all[hesap_sinifi_col] = df_tahakkuk_all[hesap_sinifi_col].str.strip()

print(df_tahakkuk_all[tarih_sutunlari].dtypes)
print("\nTarih sutunlarindaki eksik degerler:")
print(df_tahakkuk_all[tarih_sutunlari].isna().sum())
print("\nIlceler:", df_tahakkuk_all["ilce"].unique())
print("Benzersiz hesap sinifi sayisi:", df_tahakkuk_all[hesap_sinifi_col].nunique())


mali_yil_donem    datetime64[us]
fatura_tarihi     datetime64[us]
kayit_tarihi      datetime64[us]
vade_tarihi       datetime64[us]
dtype: object

Tarih sutunlarindaki eksik degerler:
mali_yil_donem    0
fatura_tarihi     0
kayit_tarihi      0
vade_tarihi       0
dtype: int64

Ilceler: <StringArray>
['HAMAMÖZÜ', 'GÜMÜŞHACIKÖY', 'GÖYNÜCEK']
Length: 3, dtype: str
Benzersiz hesap sinifi sayisi: 37


## 5. Ilce Bazinda Benzersiz Musteri Sayilari

Tahakkuk verilerinde yer alan `sozlesme_hesap_no` degiskeni kullanilarak her ilcedeki benzersiz musteri sayisi hesaplanmistir.


In [12]:
df_musteri_sayilari = (
    df_tahakkuk_all
    .groupby("ilce")["sozlesme_hesap_no"]
    .nunique()
    .sort_values(ascending=False)
    .rename("Benzersiz Musteri Sayisi")
    .reset_index()
    .rename(columns={"ilce": "Ilce"})
)

display(df_musteri_sayilari)


,Ilce,Benzersiz Musteri Sayisi
0,GÜMÜŞHACIKÖY,18190
1,GÖYNÜCEK,7128
2,HAMAMÖZÜ,2981


### Musteri Sayisi Bulgulari

- En yuksek benzersiz musteri sayisi Gumushacikoy ilcesindedir.
- Goynucek ikinci, Hamamozu ucuncu siradadir.
- Ilceler arasinda toplam tuketim karsilastirmasi yapilirken musteri sayisi farki dikkate alinmalidir. Bu nedenle sonraki analizlerde kayit basina veya musteri basina tuketim gostergeleri de onemlidir.


## 6. kWh Tuketim Verisinin Kalite Kontrolu

Birlestirilmis tahakkuk verisindeki `kwh` degiskeni eksik, negatif, sifir ve pozitif degerler acisindan incelenmistir.


In [13]:
toplam_kayit = len(df_tahakkuk_all)
eksik_kwh = df_tahakkuk_all["kwh"].isna().sum()
negatif_kwh = (df_tahakkuk_all["kwh"] < 0).sum()
sifir_kwh = (df_tahakkuk_all["kwh"] == 0).sum()
pozitif_kwh = (df_tahakkuk_all["kwh"] > 0).sum()

print(f"Toplam kayit sayisi : {toplam_kayit:,}")
print(f"Eksik kWh           : {eksik_kwh:,}")
print(f"Negatif kWh         : {negatif_kwh:,}")
print(f"Sifir kWh           : {sifir_kwh:,}")
print(f"Pozitif kWh         : {pozitif_kwh:,}")
print("-" * 40)
print(f"Minimum kWh         : {df_tahakkuk_all['kwh'].min():,.2f}")
print(f"Maksimum kWh        : {df_tahakkuk_all['kwh'].max():,.2f}")


Toplam kayit sayisi : 1,185,698
Eksik kWh           : 0
Negatif kWh         : 151
Sifir kWh           : 55,377
Pozitif kWh         : 1,130,170
----------------------------------------
Minimum kWh         : -25,370.64
Maksimum kWh        : 153,575.73


In [14]:
df_negatif_kwh = df_tahakkuk_all[df_tahakkuk_all["kwh"] < 0].copy()

negatif_ilce = (
    df_negatif_kwh["ilce"]
    .value_counts()
    .rename_axis("Ilce")
    .reset_index(name="Negatif Kayit Sayisi")
)

negatif_hesap_sinifi = (
    df_negatif_kwh[hesap_sinifi_col]
    .value_counts()
    .rename_axis("Hesap Sinifi")
    .reset_index(name="Negatif Kayit Sayisi")
)

display(negatif_ilce)
display(negatif_hesap_sinifi.head(10))


,Ilce,Negatif Kayit Sayisi
0,GÜMÜŞHACIKÖY,107
1,GÖYNÜCEK,40
2,HAMAMÖZÜ,4


,Hesap Sinifi,Negatif Kayit Sayisi
0,Mesken,106
1,Tarımsal Faaliyetler (Kooperatif),16
2,Ticari Faaliyet - Yazıhane,13
3,Tarımsal Faaliyetler (Şahıs),10
4,Şantiye ve Geçici Aboneler,2
5,Resmi Daire,2
6,Belediye,1
7,Süt Toplama Merkezi,1


In [15]:
display(
    df_negatif_kwh[
        ["ilce", "sozlesme_hesap_no", "mali_yil_donem", hesap_sinifi_col, "kwh"]
    ]
    .sort_values("kwh")
    .head(10)
)


,ilce,sozlesme_hesap_no,mali_yil_donem,Hesap Sınıfı,kwh
642557,GÜMÜŞHACIKÖY,3798287663,2025-04-01,Belediye,"-25,370.64"
614593,GÜMÜŞHACIKÖY,5966038883,2023-08-01,Tarımsal Faaliyetler (Kooperatif),"-12,574.78"
408766,GÜMÜŞHACIKÖY,6199137693,2024-08-01,Tarımsal Faaliyetler (Şahıs),"-9,069.13"
398719,GÜMÜŞHACIKÖY,3971569485,2024-08-01,Tarımsal Faaliyetler (Şahıs),"-6,851.44"
1113942,GÖYNÜCEK,7513817453,2025-03-01,Tarımsal Faaliyetler (Kooperatif),"-4,208.64"
866036,GÜMÜŞHACIKÖY,1822905563,2024-10-01,Ticari Faaliyet - Yazıhane,"-3,861.20"
1151085,GÖYNÜCEK,5811626546,2024-10-01,Mesken,"-2,836.64"
866037,GÜMÜŞHACIKÖY,1822905563,2024-10-01,Ticari Faaliyet - Yazıhane,"-2,332.63"
866030,GÜMÜŞHACIKÖY,1822905563,2024-09-01,Ticari Faaliyet - Yazıhane,"-2,315.57"
719646,GÜMÜŞHACIKÖY,9953326179,2024-10-01,Ticari Faaliyet - Yazıhane,"-1,981.66"


### Negatif Tuketim Bulgulari

- Negatif `kwh` kayitlari toplam veri icinde cok dusuk orandadir.
- Negatif degerler otomatik olarak silinmemistir; sayac duzeltmesi, mahsuplasma veya operasyonel bir surecle iliskili olabilir.
- Bu kayitlar sonraki analizlerde ihtiyac halinde ayri filtrelenebilir.


### Negatif ve Sifir Tuketimleri Nasil Yorumluyoruz

Negatif ve sifir `kwh` degerleri dogrudan silinmeden once veri icindeki dagilimlari incelenmistir. Amac, bu kayitlarin rastgele bir veri bozulmasi mi yoksa belirli abone/donem gruplarinda yogunlasan operasyonel kayitlar mi oldugunu anlamaktir.


In [16]:
sifir_kwh_df = df_tahakkuk_all[df_tahakkuk_all["kwh"] == 0].copy()

negatif_sifir_ozet = pd.DataFrame({
    "Gosterge": ["Negatif kWh", "Sifir kWh"],
    "Kayit Sayisi": [len(df_negatif_kwh), len(sifir_kwh_df)],
    "Toplam Veriye Oran (%)": [
        len(df_negatif_kwh) / len(df_tahakkuk_all) * 100,
        len(sifir_kwh_df) / len(df_tahakkuk_all) * 100,
    ],
    "Benzersiz Musteri": [
        df_negatif_kwh["sozlesme_hesap_no"].nunique(),
        sifir_kwh_df["sozlesme_hesap_no"].nunique(),
    ],
}).round(4)

display(negatif_sifir_ozet)

print("Negatif kWh - donem dagilimi")
display(
    df_negatif_kwh["mali_yil_donem"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .rename_axis("Donem")
    .reset_index(name="Kayit Sayisi")
    .tail(12)
)

print("Sifir kWh - ilk 10 hesap sinifi")
display(
    sifir_kwh_df[hesap_sinifi_col]
    .value_counts()
    .rename_axis("Hesap Sinifi")
    .reset_index(name="Sifir Kayit Sayisi")
    .head(10)
)

print("Sifir kWh - ilce dagilimi")
display(
    sifir_kwh_df["ilce"]
    .value_counts()
    .rename_axis("Ilce")
    .reset_index(name="Sifir Kayit Sayisi")
)


,Gosterge,Kayit Sayisi,Toplam Veriye Oran (%),Benzersiz Musteri
0,Negatif kWh,151,0.01,54
1,Sifir kWh,55377,4.67,9883


Negatif kWh - donem dagilimi


,Donem,Kayit Sayisi
18,2024-07,8
19,2024-08,8
20,2024-09,5
21,2024-10,10
22,2024-11,5
23,2024-12,6
24,2025-01,4
25,2025-02,1
26,2025-03,4
27,2025-04,4


Sifir kWh - ilk 10 hesap sinifi


,Hesap Sinifi,Sifir Kayit Sayisi
0,Mesken,47986
1,Ticari Faaliyet - Yazıhane,3102
2,Tarımsal Faaliyetler (Şahıs),1632
3,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,372
4,Şantiye ve Geçici Aboneler,344
5,İbadethane Isıtma/Soğutma/Lojman,323
6,Tarımsal Faaliyetler (Kooperatif),303
7,Köy İçme Suyu Temini ve Dağıtımı Tesisi,223
8,Belediye,150
9,Resmi Daire Lojman,149


Sifir kWh - ilce dagilimi


,Ilce,Sifir Kayit Sayisi
0,GÜMÜŞHACIKÖY,31932
1,GÖYNÜCEK,16824
2,HAMAMÖZÜ,6621


## 7. Aykiri Deger Analizi

`kwh` tuketim degerlerindeki potansiyel aykiri gozlemler ceyrekler acikligi (IQR) yontemi ile incelenmistir.


In [17]:
Q1 = df_tahakkuk_all["kwh"].quantile(0.25)
Q3 = df_tahakkuk_all["kwh"].quantile(0.75)
IQR = Q3 - Q1

alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

df_outliers = df_tahakkuk_all[
    (df_tahakkuk_all["kwh"] < alt_sinir) |
    (df_tahakkuk_all["kwh"] > ust_sinir)
].copy()

print(f"Q1 (25. yuzdelik)      : {Q1:,.2f} kWh")
print(f"Q3 (75. yuzdelik)      : {Q3:,.2f} kWh")
print(f"IQR                    : {IQR:,.2f} kWh")
print(f"Alt sinir              : {alt_sinir:,.2f} kWh")
print(f"Ust sinir              : {ust_sinir:,.2f} kWh")
print("-" * 45)
print(f"Aykiri deger sayisi    : {len(df_outliers):,}")
print(f"Aykiri deger orani     : %{len(df_outliers) / len(df_tahakkuk_all) * 100:.2f}")


Q1 (25. yuzdelik)      : 18.01 kWh
Q3 (75. yuzdelik)      : 80.00 kWh
IQR                    : 61.99 kWh
Alt sinir              : -74.97 kWh
Ust sinir              : 172.98 kWh
---------------------------------------------
Aykiri deger sayisi    : 48,554
Aykiri deger orani     : %4.09


In [18]:
outlier_ilce = (
    df_outliers["ilce"]
    .value_counts()
    .rename_axis("Ilce")
    .reset_index(name="Aykiri Kayit Sayisi")
)

outlier_hesap_sinifi = (
    df_outliers[hesap_sinifi_col]
    .value_counts()
    .rename_axis("Hesap Sinifi")
    .reset_index(name="Aykiri Kayit Sayisi")
)

display(outlier_ilce)
display(outlier_hesap_sinifi.head(10))


,Ilce,Aykiri Kayit Sayisi
0,GÜMÜŞHACIKÖY,31838
1,GÖYNÜCEK,12358
2,HAMAMÖZÜ,4358


,Hesap Sinifi,Aykiri Kayit Sayisi
0,Mesken,20922
1,Ticari Faaliyet - Yazıhane,15818
2,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,2260
3,Tarımsal Faaliyetler (Şahıs),1896
4,Köy İçme Suyu Temini ve Dağıtımı Tesisi,1315
5,Tarımsal Faaliyetler (Kooperatif),997
6,İbadethane Isıtma/Soğutma/Lojman,924
7,Belediye,726
8,Resmi Daire,636
9,Süt Toplama Merkezi,624


In [19]:
display(
    df_tahakkuk_all[
        ["ilce", "sozlesme_hesap_no", "mali_yil_donem", hesap_sinifi_col, "kwh"]
    ]
    .sort_values("kwh", ascending=False)
    .head(10)
)


,ilce,sozlesme_hesap_no,mali_yil_donem,Hesap Sınıfı,kwh
808462,GÜMÜŞHACIKÖY,6414845714,2024-12-01,Lisansız Üreticiler,"153,575.73"
808466,GÜMÜŞHACIKÖY,6414845714,2025-02-01,Lisansız Üreticiler,"150,757.74"
808470,GÜMÜŞHACIKÖY,6414845714,2025-04-01,Lisansız Üreticiler,"136,527.93"
808464,GÜMÜŞHACIKÖY,6414845714,2025-01-01,Lisansız Üreticiler,"134,681.40"
808448,GÜMÜŞHACIKÖY,6414845714,2024-05-01,Lisansız Üreticiler,"124,008.57"
808458,GÜMÜŞHACIKÖY,6414845714,2024-10-01,Lisansız Üreticiler,"119,221.20"
808472,GÜMÜŞHACIKÖY,6414845714,2025-05-01,Lisansız Üreticiler,"116,408.88"
808468,GÜMÜŞHACIKÖY,6414845714,2025-03-01,Lisansız Üreticiler,"111,801.06"
721105,GÜMÜŞHACIKÖY,9150855017,2023-02-01,Sanayi,"109,546.29"
721103,GÜMÜŞHACIKÖY,9150855017,2023-01-01,Sanayi,"109,344.06"


### Aykiri Deger Bulgulari

- IQR yontemi potansiyel aykiri tuketimleri isaretlemek icin kullanilmistir.
- Yuksek tuketimler ozellikle sanayi, lisansiz uretici, belediye ve altyapi odakli hesap siniflarinda dogal olabilir.
- Bu nedenle aykiri degerler otomatik olarak veri setinden cikarilmamis, baglamsal olarak degerlendirilmek uzere korunmustur.


## 8. Hesap Sinifina Gore Tuketim Istatistikleri

Farkli musteri gruplarinin elektrik tuketim profillerini karsilastirmak icin hesap sinifi bazinda temel istatistikler hesaplanmistir.


In [20]:
hesap_sinifi_istatistikleri = (
    df_tahakkuk_all
    .groupby(hesap_sinifi_col)["kwh"]
    .agg(
        Kayit_Sayisi="count",
        Ortalama_kWh="mean",
        Medyan_kWh="median",
        Standart_Sapma_kWh="std"
    )
    .sort_values("Ortalama_kWh", ascending=False)
    .reset_index()
)

hesap_sinifi_istatistikleri[
    ["Ortalama_kWh", "Medyan_kWh", "Standart_Sapma_kWh"]
] = hesap_sinifi_istatistikleri[
    ["Ortalama_kWh", "Medyan_kWh", "Standart_Sapma_kWh"]
].round(2)

display(hesap_sinifi_istatistikleri)


,Hesap Sınıfı,Kayit_Sayisi,Ortalama_kWh,Medyan_kWh,Standart_Sapma_kWh
0,Karayolları Genel Müdürlüğü Aydınlatma,41,"30,203.43","36,810.90","16,963.57"
1,Aritma Tesisleri,35,"16,594.17","16,186.91","11,656.71"
2,Lisansız Üreticiler,180,"16,155.25",322.32,"35,926.42"
3,Sanayi,187,"7,293.80","1,531.68","13,716.34"
4,İçme-Kullanma Suyu (Belediye),417,"5,213.51",400.02,"8,948.07"
5,Tarımsal Faaliyetler (Kooperatif),1632,"3,779.29",282.72,"9,426.94"
6,Lisansız Üreticiler - Resmi Daire,36,"1,195.05",770.62,"1,229.58"
7,"Resmi SAĞLIK KURULUŞLARI,RESMİ SPOR TES.",442,"1,154.74",239.90,"3,804.87"
8,"Resmi Üniversite,Yük.Okul,Kurs,Yurt,Okul",476,883.89,285.41,"1,435.31"
9,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,8500,688.44,23.86,"3,911.91"


### Hesap Sinifi Bulgulari

- Mesken sinifi kayit sayisi bakimindan veri setinin buyuk bolumunu olusturmaktadir.
- Karayollari aydinlatma, aritma tesisleri, lisansiz ureticiler ve sanayi gibi siniflarda kayit basina ortalama tuketim cok daha yuksektir.
- Bazi siniflarda ortalama ve medyan arasindaki fark buyuktur; bu durum dagilimin yuksek tuketimlere dogru carpik oldugunu gosterir.
- Az sayida gozleme sahip hesap siniflarinda ortalama tuketim yorumlanirken orneklem buyuklugu dikkate alinmalidir.


## 9. Analiz Kapsami Kontrolu

Tahakkuk sayfalari Hamamozu, Gumushacikoy ve Goynucek ilcelerini kapsamaktadir. Tahsilat verilerinde bu kapsamin disinda ilce bulunup bulunmadigi ayrica kontrol edilmistir. Eslesen tahakkuk verisi olmayan ilceler, karsilastirmali analizlerde kapsam disi birakilacaktir.


In [21]:
hedef_ilceler = sorted(df_tahakkuk_all["ilce"].dropna().astype("string").str.strip().unique())

tahsilat_ilce_col = df_tahsilat.columns[2]
tahsilat1_ilce_col = df_tahsilat_1.columns[2]

tahsilat_ilceleri = sorted(df_tahsilat[tahsilat_ilce_col].dropna().astype("string").str.strip().unique())
tahsilat1_ilceleri = sorted(df_tahsilat_1[tahsilat1_ilce_col].dropna().astype("string").str.strip().unique())

kapsam_disi_tahsilat = sorted(set(tahsilat_ilceleri) - set(hedef_ilceler))
kapsam_disi_tahsilat_1 = sorted(set(tahsilat1_ilceleri) - set(hedef_ilceler))

print("Analiz kapsamina dahil edilen ilceler:", hedef_ilceler)
print("Tahsilat sayfasinda kapsam disi ilceler:", kapsam_disi_tahsilat)
print("Tahsilat 1 sayfasinda kapsam disi ilceler:", kapsam_disi_tahsilat_1)


Analiz kapsamina dahil edilen ilceler: ['GÖYNÜCEK', 'GÜMÜŞHACIKÖY', 'HAMAMÖZÜ']
Tahsilat sayfasinda kapsam disi ilceler: ['TAŞOVA']
Tahsilat 1 sayfasinda kapsam disi ilceler: ['TAŞOVA']


### Kapsam Bulgusu

Tahsilat sayfasinda tahakkuk verileriyle eslesmeyen Tasova kayitlari bulunmaktadir. Bu ilce icin tahakkuk verisi olmadigindan, tuketim-tahsilat karsilastirmalarinda yanli sonuc uretmemek icin Tasova kayitlari analiz kopyalarindan cikarilmistir. Ham veri ise korunmustur.


## 10. Analize Hazirlik

Ham veri setleri korunmus, sonraki notebooklarda kullanilmak uzere analiz kopyalari olusturulmustur. Tahakkuk kapsaminda olmayan ilceler analiz kopyalarindan cikarilmis; odeme turu ve odeme zamanlamasi sutunlarindaki eksik degerler ilgili kategoride tahsilat bulunmadigi anlamina geldigi icin `0` ile doldurulmustur.


In [22]:
df_tahsilat_clean = df_tahsilat.copy()
df_tahsilat_1_clean = df_tahsilat_1.copy()
df_tahakkuk_clean = df_tahakkuk_all.copy()

tahsilat_ilce_col = df_tahsilat_clean.columns[2]
tahsilat1_ilce_col = df_tahsilat_1_clean.columns[2]

df_tahsilat_clean[tahsilat_ilce_col] = df_tahsilat_clean[tahsilat_ilce_col].astype("string").str.strip()
df_tahsilat_1_clean[tahsilat1_ilce_col] = df_tahsilat_1_clean[tahsilat1_ilce_col].astype("string").str.strip()

df_tahsilat_clean = df_tahsilat_clean[df_tahsilat_clean[tahsilat_ilce_col].isin(hedef_ilceler)].copy()
df_tahsilat_1_clean = df_tahsilat_1_clean[df_tahsilat_1_clean[tahsilat1_ilce_col].isin(hedef_ilceler)].copy()

print("Filtre sonrasi tahsilat ilceleri:", sorted(df_tahsilat_clean[tahsilat_ilce_col].dropna().unique()))
print("Filtre sonrasi Tahsilat 1 ilceleri:", sorted(df_tahsilat_1_clean[tahsilat1_ilce_col].dropna().unique()))

odeme_turu_sutunlari = df_tahsilat_clean.columns[5:9].tolist()
odeme_zamani_sutunlari = df_tahsilat_1_clean.columns[6:22].tolist()

df_tahsilat_clean[odeme_turu_sutunlari] = df_tahsilat_clean[odeme_turu_sutunlari].fillna(0)
df_tahsilat_1_clean[odeme_zamani_sutunlari] = df_tahsilat_1_clean[odeme_zamani_sutunlari].fillna(0)


Filtre sonrasi tahsilat ilceleri: ['GÖYNÜCEK', 'GÜMÜŞHACIKÖY', 'HAMAMÖZÜ']
Filtre sonrasi Tahsilat 1 ilceleri: ['GÖYNÜCEK', 'GÜMÜŞHACIKÖY', 'HAMAMÖZÜ']


In [23]:
ay_haritasi = {
    "OCK": 1,
    "SBT": 2,
    "SUB": 2,
    "MAR": 3,
    "NIS": 4,
    "MAY": 5,
    "HAZ": 6,
    "TEM": 7,
    "AGU": 8,
    "EYL": 9,
    "EKM": 10,
    "KAS": 11,
    "KSM": 11,
    "ARA": 12,
    "ARL": 12,
}

def turkce_karakterleri_sadelestir(seri):
    return (
        seri.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(chr(350), "S", regex=False)
        .str.replace(chr(286), "G", regex=False)
        .str.replace(chr(220), "U", regex=False)
        .str.replace(chr(214), "O", regex=False)
        .str.replace(chr(304), "I", regex=False)
        .str.replace(chr(199), "C", regex=False)
    )

def turkce_donem_to_datetime(seri):
    parcalar = turkce_karakterleri_sadelestir(seri).str.split(expand=True)
    ay = parcalar[0].map(ay_haritasi)
    yil = pd.to_numeric(parcalar[1], errors="coerce")
    return pd.to_datetime({"year": yil, "month": ay, "day": 1}, errors="coerce")

donem_col = df_tahsilat_1_clean.columns[0]

df_tahsilat_clean["Tahsilat Tarihi"] = pd.to_datetime(
    df_tahsilat_clean["Tahsilat Tarihi"],
    errors="coerce"
)

df_tahsilat_1_clean["donem_tarihi"] = turkce_donem_to_datetime(
    df_tahsilat_1_clean[donem_col]
)

print("Tahsilat tarihi eksik deger:", df_tahsilat_clean["Tahsilat Tarihi"].isna().sum())
print("Donem tarihi eksik deger   :", df_tahsilat_1_clean["donem_tarihi"].isna().sum())


Tahsilat tarihi eksik deger: 0
Donem tarihi eksik deger   : 0


In [24]:
duplicate_count = df_tahakkuk_clean.duplicated().sum()

print(f"Tahakkuk tamamen yinelenen kayit sayisi: {duplicate_count:,}")
print(f"Yinelenen kayit orani: %{duplicate_count / len(df_tahakkuk_clean) * 100:.4f}")
print("\nTemiz kopyalarda kalan toplam eksik degerler:")
print("Tahsilat  :", df_tahsilat_clean.isna().sum().sum())
print("Tahsilat 1:", df_tahsilat_1_clean.isna().sum().sum())
print("Tahakkuk  :", df_tahakkuk_clean.isna().sum().sum())


Tahakkuk tamamen yinelenen kayit sayisi: 0
Yinelenen kayit orani: %0.0000

Temiz kopyalarda kalan toplam eksik degerler:
Tahsilat  : 0
Tahsilat 1: 0
Tahakkuk  : 0


## 11. Genel Degerlendirme

- Uc ilceye ait tahakkuk verileri basariyla birlestirilmis ve toplam kayit sayisi dogrulanmistir.
- Gumushacikoy musteri ve kayit hacmi bakimindan en buyuk ilcedir; bu nedenle toplam tuketim yorumlarinda ilce buyuklugu dikkate alinmalidir.
- `kwh` degiskeninde eksik deger bulunmamaktadir; buna karsin negatif ve sifir tuketim kayitlari tespit edilmistir.
- Negatif tuketimler fiziksel tuketim davranisi olarak yorumlanmamalidir. Ancak veri icindeki sayilari cok dusuk oldugu ve belirli kayitlarda yogunlastigi icin bu degerler dogrudan silinmemis, operasyonel duzeltme/iptal/mahsuplasma ihtimali olan kayitlar olarak isaretlenmistir.
- Sifir tuketimler negatif degerlerden farkli olarak gercek bir musteri durumunu temsil edebilir. Bu nedenle sifir kayitlar da silinmemis, hangi ilce ve hesap siniflarinda yogunlastigi ayrica incelenmistir.
- Bu yaklasimla ham veri korunmus, negatif ve sifir tuketimler ise sonraki analizlerde dahil/haric senaryolar icin ayrilabilecek veri kalitesi isaretleri olarak ele alinmistir.
- IQR yontemiyle potansiyel aykiri degerler belirlenmis ancak hesap siniflari arasindaki dogal tuketim farklari nedeniyle otomatik olarak silinmemistir.
- Tahsilat sayfasinda tahakkuk kapsaminda olmayan Tasova kayitlari tespit edilmis; eslesen tahakkuk verisi olmadigi icin sonraki analizlerde kapsam disi birakilmistir.
- Tuketim davranisi hesap siniflarina gore belirgin sekilde degismektedir.
- Hazirlanan temiz kopyalar sonraki gorsellestirme ve veri hikayesi notebooklari icin temel olusturacaktir.
